# HW13: BERT Tokenization, Inference, and Fine-tuning

## 1. Imports and Setup

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## 2. Data Loading and Analysis

In [ ]:
dataset = load_dataset('emotion')
print(f'Train size: {len(dataset["train"])}')
print(f'Validation size: {len(dataset["validation"])}')
print(f'Test size: {len(dataset["test"])}')

label_names = dataset['train'].features['label'].names
print(f'\nClasses: {label_names}')
print(f'Number of classes: {len(label_names)}')

In [ ]:
print('Sample texts and labels:')
for i in range(5):
    sample = dataset['train'][i]
    print(f"{i+1}. [{label_names[sample['label']]}] {sample['text'][:80]}...")

## 3. Tokenization Analysis

In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f'Tokenizer: {model_name}')
print(f'Vocabulary size: {tokenizer.vocab_size}')
print(f'Special tokens: {tokenizer.special_tokens_map}')

In [ ]:
sample_texts = dataset['train'][:3]['text']

for idx, text in enumerate(sample_texts):
    print(f'\n--- Sample {idx+1} ---')
    print(f'Text: {text[:60]}...')
    
    tokens = tokenizer.tokenize(text)
    print(f'Tokens: {tokens[:15]}...')
    
    encoded = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    print(f'Input IDs shape: {encoded["input_ids"].shape}')
    print(f'Attention mask shape: {encoded["attention_mask"].shape}')
    print(f'Tokens count: {encoded["input_ids"].shape[1]}')

In [ ]:
short_text = "Great movie!"
long_text = " ".join(["word"] * 200)

print('Padding and truncation demo:')
print(f'\nShort text tokens: {tokenizer(short_text, max_length=10, padding="max_length")["input_ids"]}')
print(f'Long text (truncated): {tokenizer(long_text, max_length=20, truncation=True)["input_ids"]}')

## 4. Pretrained Model Inference

In [ ]:
pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(label_names)
).to(DEVICE)
pretrained_model.eval()

print(f'Model loaded: {model_name}')

In [ ]:
test_texts = [
    "I love this so much!",
    "This is terrible and makes me angry",
    "I feel so sad today",
    "That's cool and interesting",
    "I'm shocked by this news"
]

print('Pretrained model inference on sample texts:')
with torch.no_grad():
    for text in test_texts:
        inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
        outputs = pretrained_model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        pred_label = torch.argmax(probs, dim=-1).item()
        confidence = probs[0, pred_label].item()
        
        print(f'Text: "{text}"')
        print(f'Prediction: {label_names[pred_label]} (confidence: {confidence:.2f})')
        print()

## 5. Data Preparation for Fine-tuning

In [ ]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )

dataset_tokenized = dataset.map(tokenize_function, batched=True)
dataset_tokenized = dataset_tokenized.remove_columns('text')
dataset_tokenized.set_format('torch')

print(f'Tokenized dataset prepared')
print(f'Train: {len(dataset_tokenized["train"])}')
print(f'Validation: {len(dataset_tokenized["validation"])}')
print(f'Test: {len(dataset_tokenized["test"])}')

## 6. Fine-tuning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_names)
)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    return {'accuracy': accuracy, 'f1_macro': f1_macro}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenized['train'],
    eval_dataset=dataset_tokenized['validation'],
    compute_metrics=compute_metrics,
)

print('Starting fine-tuning...')
trainer.train()

## 7. Evaluation on Test Set

In [ ]:
eval_results = trainer.evaluate(dataset_tokenized['test'])
print(f'Test Accuracy: {eval_results["eval_accuracy"]:.4f}')
print(f'Test F1 Macro: {eval_results["eval_f1_macro"]:.4f}')

## 8. Predictions and Analysis

In [ ]:
predictions_output = trainer.predict(dataset_tokenized['test'])
logits = predictions_output.predictions
true_labels = predictions_output.label_ids

probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
pred_labels = np.argmax(logits, axis=1)

test_texts_full = dataset['test']['text']

predictions_df = pd.DataFrame({
    'text': test_texts_full,
    'true_label': [label_names[i] for i in true_labels],
    'pred_label': [label_names[i] for i in pred_labels],
    'confidence': np.max(probs, axis=1)
})

predictions_df.to_csv('HW13/artifacts/sample_predictions.csv', index=False)
print(f'Saved {len(predictions_df)} predictions')
print(predictions_df.head(10))

## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(true_labels, pred_labels)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix on Test Set')
plt.tight_layout()
plt.savefig('HW13/artifacts/confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print('Confusion matrix saved')

## 10. Error Analysis

In [ ]:
errors = predictions_df[predictions_df['true_label'] != predictions_df['pred_label']]
print(f'Total errors: {len(errors)} out of {len(predictions_df)}')
print(f'Error rate: {len(errors)/len(predictions_df)*100:.1f}%')

print('\nSample errors:')
for idx, row in errors.head(10).iterrows():
    print(f"Text: {row['text'][:60]}...")
    print(f"True: {row['true_label']}, Pred: {row['pred_label']}, Conf: {row['confidence']:.2f}")
    print()

## 11. Metrics Summary

In [ ]:
accuracy = accuracy_score(true_labels, pred_labels)
f1_macro = f1_score(true_labels, pred_labels, average='macro')
f1_per_class = f1_score(true_labels, pred_labels, average=None)

print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test F1 Macro: {f1_macro:.4f}')
print(f'\nF1 per class:')
for label, f1 in zip(label_names, f1_per_class):
    print(f'  {label}: {f1:.4f}')